# ImageNet Sampling for User Study

Create a 100-class subset for the user study by mapping ImageNet synsets such as `n01843383` to numeric class folders such as `class_0999`, then copying the first 3 images from each selected class into an output directory.

## Objective

This notebook does four things:

- loads the 100 target ImageNet synsets from `imagenet_miniclasses.txt`
- converts each synset to your numeric class id (`class_####`)
- copies the first 3 images from each chosen class folder
- writes a manifest CSV describing what was selected

Update the config cell below before running.

In [ ]:
from __future__ import annotations

import ast
import csv
import json
import shutil
from pathlib import Path
from typing import Iterable


In [ ]:
PROJECT_DIR = Path("/Users/mrmc/Documents/GitHub/DuoDiT-UserStudy")
SYNSET_LIST_PATH = PROJECT_DIR / "imagenet_miniclasses.txt"

# Set this to the root that contains folders such as samples/class_0999/
SAMPLE_ROOT = Path("/path/to/your/samples")

# Set this to an ImageNet class map. Supported formats:
# 1. Keras imagenet_class_index.json
# 2. JSON mapping index -> synset
# 3. text file with one synset per line in ImageNet class order
CLASS_MAP_PATH = Path("/path/to/imagenet_class_index.json")

# Output directory for the selected subset
OUTPUT_DIR = PROJECT_DIR / "selected_user_study_samples"
MANIFEST_CSV = OUTPUT_DIR / "selection_manifest.csv"
IMAGES_PER_CLASS = 3
EXPECTED_CLASS_COUNT = 100


In [ ]:
def load_target_synsets(path: Path) -> list[str]:
    text = path.read_text(encoding="utf-8").strip()
    data = ast.literal_eval(text)
    if not isinstance(data, list):
        raise TypeError(f"Expected a list in {path}, got {type(data).__name__}")
    synsets = [str(item).strip() for item in data]
    return synsets


def load_index_to_synset(path: Path) -> dict[int, str]:
    if path.suffix.lower() == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict):
            raise TypeError("JSON class map must be an object/dict")

        sample_value = next(iter(data.values()))
        if isinstance(sample_value, list) and sample_value:
            # Keras format: {"0": ["n01440764", "tench"], ...}
            return {int(k): str(v[0]) for k, v in data.items()}

        return {int(k): str(v) for k, v in data.items()}

    synsets = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        synsets.append(line)
    return {idx: synset for idx, synset in enumerate(synsets)}


def invert_index_map(index_to_synset: dict[int, str]) -> dict[str, int]:
    synset_to_index = {}
    for idx, synset in index_to_synset.items():
        if synset in synset_to_index:
            raise ValueError(f"Duplicate synset in class map: {synset}")
        synset_to_index[synset] = idx
    return synset_to_index


def class_dir_name(index: int) -> str:
    return f"class_{index:04d}"


In [ ]:
target_synsets = load_target_synsets(SYNSET_LIST_PATH)
index_to_synset = load_index_to_synset(CLASS_MAP_PATH)
synset_to_index = invert_index_map(index_to_synset)

print(f"Loaded {len(target_synsets)} target synsets")
print(f"Loaded {len(index_to_synset)} class-map entries")

missing_synsets = [syn for syn in target_synsets if syn not in synset_to_index]
print(f"Synsets missing from class map: {len(missing_synsets)}")
if missing_synsets:
    missing_synsets[:10]


In [ ]:
selected_classes = []
for synset in target_synsets:
    if synset not in synset_to_index:
        continue
    idx = synset_to_index[synset]
    selected_classes.append({
        "synset": synset,
        "class_index": idx,
        "class_dir": class_dir_name(idx),
    })

print(f"Resolved {len(selected_classes)} classes")
selected_classes[:5]


In [ ]:
def list_class_images(class_path: Path) -> list[Path]:
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    return sorted(path for path in class_path.iterdir() if path.is_file() and path.suffix.lower() in exts)


def build_selection_plan(sample_root: Path, classes: Iterable[dict], images_per_class: int) -> list[dict]:
    plan = []
    for item in classes:
        class_path = sample_root / item["class_dir"]
        if not class_path.exists():
            plan.append({**item, "class_path": class_path, "status": "missing_dir", "selected": []})
            continue

        images = list_class_images(class_path)
        chosen = images[:images_per_class]
        status = "ok" if len(chosen) == images_per_class else f"only_{len(chosen)}_images"
        plan.append({**item, "class_path": class_path, "status": status, "selected": chosen})
    return plan


selection_plan = build_selection_plan(SAMPLE_ROOT, selected_classes, IMAGES_PER_CLASS)
status_counts = {}
for row in selection_plan:
    status_counts[row["status"]] = status_counts.get(row["status"], 0) + 1
status_counts


In [ ]:
preview_rows = []
for row in selection_plan[:10]:
    preview_rows.append({
        "synset": row["synset"],
        "class_dir": row["class_dir"],
        "status": row["status"],
        "selected_count": len(row["selected"]),
    })
preview_rows


In [ ]:
def copy_selection(plan: list[dict], output_dir: Path, manifest_csv: Path) -> list[dict]:
    output_dir.mkdir(parents=True, exist_ok=True)
    copied = []

    for row in plan:
        if row["status"] != "ok":
            continue

        target_class_dir = output_dir / row["class_dir"]
        target_class_dir.mkdir(parents=True, exist_ok=True)

        for order, src_path in enumerate(row["selected"], start=1):
            dst_path = target_class_dir / src_path.name
            shutil.copy2(src_path, dst_path)
            copied.append({
                "synset": row["synset"],
                "class_index": row["class_index"],
                "class_dir": row["class_dir"],
                "copy_order": order,
                "source_path": str(src_path),
                "target_path": str(dst_path),
            })

    with manifest_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["synset", "class_index", "class_dir", "copy_order", "source_path", "target_path"],
        )
        writer.writeheader()
        writer.writerows(copied)

    return copied


In [ ]:
# Run this cell after confirming SAMPLE_ROOT and CLASS_MAP_PATH.
# It copies the first 3 images from each resolved class into OUTPUT_DIR/class_####/.

# copied_rows = copy_selection(selection_plan, OUTPUT_DIR, MANIFEST_CSV)
# print(f"Copied {len(copied_rows)} images into {OUTPUT_DIR}")
# print(f"Manifest written to {MANIFEST_CSV}")


## Validation

After copying, verify:

- `len(copied_rows) == 300` for a complete 100-class x 3-image subset
- each copied image lives under `OUTPUT_DIR/class_####/`
- `selection_manifest.csv` contains one row per copied image

If some classes are missing, inspect `status_counts` and the `selection_plan` preview to see which source directories or mappings need adjustment.